# Window Functions

Window functions perform calculations across rows related to the current row.

## Learning Objectives

- Understand window specifications
- Use ranking functions
- Apply analytic functions (lead, lag)
- Calculate running totals and moving averages

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, rank, dense_rank, lead, lag, sum, avg
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Window-Functions") \
    .getOrCreate()

## 1. Window Specification Basics

A window specification defines which rows are included in the calculation.

In [ ]:
# Create sample data
sales = [
    ("2024-01-01", "Alice", 1000),
    ("2024-01-01", "Bob", 1500),
    ("2024-01-02", "Alice", 1200),
    ("2024-01-02", "Bob", 800),
    ("2024-01-03", "Alice", 900),
    ("2024-01-03", "Bob", 1100),
    ("2024-01-04", "Alice", 1300),
    ("2024-01-04", "Bob", 950),
]

df = spark.createDataFrame(sales, ["date", "salesperson", "amount"])
df.show()

In [ ]:
# Define a window partitioned by salesperson, ordered by date
window_spec = Window.partitionBy("salesperson").orderBy("date")

print("Window specification created!")

## 2. Ranking Functions

Ranking functions assign a rank to each row within a partition.

In [ ]:
# row_number: unique sequential number
df.withColumn("row_num", row_number().over(window_spec)).show()

In [ ]:
# rank: same rank for ties, gaps in ranking
window_by_amount = Window.partitionBy("salesperson").orderBy(col("amount").desc())

df.withColumn("rank", rank().over(window_by_amount)).show()

In [ ]:
# dense_rank: same rank for ties, no gaps
df.withColumn("dense_rank", dense_rank().over(window_by_amount)).show()

## 3. Lead and Lag

Lead and lag access values from other rows.

In [ ]:
# Lag: get value from previous row
df.withColumn("prev_amount", lag("amount", 1).over(window_spec)).show()

In [ ]:
# Lead: get value from next row
df.withColumn("next_amount", lead("amount", 1).over(window_spec)).show()

In [ ]:
# Calculate day-over-day change
df.withColumn("prev_amount", lag("amount", 1).over(window_spec)) \
   .withColumn("change", col("amount") - col("prev_amount")) \
   .show()

## 4. Running Totals and Moving Averages

Window frames allow calculations over a range of rows.

In [ ]:
# Running total (cumulative sum)
# Rows between start and current row
running_window = Window.partitionBy("salesperson") \
    .orderBy("date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df.withColumn("running_total", sum("amount").over(running_window)).show()

In [ ]:
# Moving average (last 3 rows including current)
moving_window = Window.partitionBy("salesperson") \
    .orderBy("date") \
    .rowsBetween(-2, Window.currentRow)  # 2 rows before to current

df.withColumn("moving_avg", avg("amount").over(moving_window)).show()

In [ ]:
# Total sum across entire partition
total_window = Window.partitionBy("salesperson") \
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

df.withColumn("total", sum("amount").over(total_window)) \
   .withColumn("pct_of_total", col("amount") / col("total") * 100) \
   .show()

## 5. Top N Per Group

A common pattern is finding the top N items per group.

In [ ]:
# Find top 2 sales per salesperson
window_rank = Window.partitionBy("salesperson").orderBy(col("amount").desc())

df.withColumn("rank", row_number().over(window_rank)) \
   .filter(col("rank") <= 2) \
   .show()

## 6. Working with Sample Data

In [ ]:
try:
    orders = spark.read.parquet("/opt/spark/data/orders/small")
    
    # Rank orders by amount per user
    window_user = Window.partitionBy("user_id").orderBy(col("total_amount").desc())
    
    orders.withColumn("order_rank", row_number().over(window_user)) \
          .filter(col("order_rank") <= 3) \
          .select("order_id", "user_id", "total_amount", "order_rank") \
          .show(20)
    
except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")

## 7. Exercises

In [ ]:
# Exercise 1: Calculate the running total of orders per user
# Your code here:


In [ ]:
# Exercise 2: Find the 2nd highest order amount per user
# Your code here:


In [ ]:
# Exercise 3: Calculate the difference between each order and the user's average order
# Your code here:


In [ ]:
# Clean up
spark.stop()